In [6]:
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import r2_score , accuracy_score
from sklearn.preprocessing import OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer

# 1. Load and Clean
df = pd.read_csv("Deffective.csv")
df_clean = df.dropna(subset=['Defective_Units']).copy()

# 2. Feature Engineering
df_clean['Order_Date'] = pd.to_datetime(df_clean['Order_Date'])
df_clean['Delivery_Date'] = pd.to_datetime(df_clean['Delivery_Date'])
df_clean['Delivery_Time_Days'] = (df_clean['Delivery_Date'] - df_clean['Order_Date']).dt.days
df_clean['Discount'] = df_clean['Unit_Price'] - df_clean['Negotiated_Price']

# 3. Setup Features and Target
features = ['Supplier', 'Item_Category', 'Order_Status', 'Quantity', 'Delivery_Time_Days', 'Discount']
X = df_clean[features]
y = df_clean['Defective_Units']

# 4. Preprocessing Pipeline
preprocessor = ColumnTransformer(
    transformers=[
        ('num', SimpleImputer(strategy='median'), ['Quantity', 'Delivery_Time_Days', 'Discount']),
        ('cat', OneHotEncoder(handle_unknown='ignore'), ['Supplier', 'Item_Category', 'Order_Status'])
    ])

# 5. Train Model
model = Pipeline(steps=[('preprocessor', preprocessor),
                        ('regressor', RandomForestRegressor(n_estimators=100, random_state=42))])

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)
model.fit(X_train, y_train)

# 6. Evaluate
print(f"R-squared: {r2_score(y_test, model.predict(X_test)):.4f}")


R-squared: 0.9660
